# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show available record sets and field IDs using their @id

record_sets = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if record_sets:
    print('Record Sets in dataset:')
    for idx, rs_id in enumerate(record_sets):
        print(f"{idx+1}. @id: {rs_id}")
        # get field @id and names
        rs_obj = [r for r in metadata.to_json()['recordSet'] if r['@id'] == rs_id][0]
        fields = rs_obj.get('field', [])
        print(f"   Fields: {[f['@id'] if isinstance(f, dict) else f for f in fields]}")
else:
    print('No record sets found. Attempting to list from distributions...')
    # Sometimes data stored as single CSV referenced in distribution
    distributions = metadata.to_json().get('distribution', [])
    for dist in distributions:
        print(f"Distribution @id: {dist['@id']}")
    print('\nYou can access record sets by loading these distributions as record sets.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Since no explicit 'recordSet' is present in the metadata, we will investigate possible record sets by using distributions.

import pprint

distros = metadata.to_json().get('distribution', [])
record_sets = []
for dist in distros:
    # mlcroissant uses distribution @id as record_set id
    record_sets.append(dist['@id'])
    print(f"Possible record set: {dist['@id']} (file: {dist.get('name', dist.get('contentUrl'))})")

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        # Only nonempty dataframes
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id {record_set_id}")
        print("Columns:", df.columns.tolist())

# Pick a record set for further exploration
main_record_set = record_sets[0]
print(f"\nUsing main record set: {main_record_set}")
display(dataframes[main_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Identify numeric fields from the DataFrame columns
df = dataframes[main_record_set]

# Attempt to detect numeric fields
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

print('Numeric fields detected:', numeric_fields)
if numeric_fields:
    numeric_field = numeric_fields[0]
else:
    # Try to coerce some columns to numeric (many DTI fields are like 'FA_1', 'AD_2' etc)
    possible_numeric = [col for col in df.columns if any(substr in col.upper() for substr in ['FA', 'AD', 'MD', 'RD'])]
    for col in possible_numeric:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    numeric_field = numeric_fields[0] if numeric_fields else df.columns[0]
    print('Numeric field chosen after forced conversion:', numeric_field)

# Use sensible default threshold
threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f} (mean):")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a non-numeric field, e.g. subject or sex or diagnosis/label
group_candidates = [col for col in df.columns if (df[col].nunique() < len(df) // 3) and pd.api.types.is_object_dtype(df[col])]
if group_candidates:
    group_field = group_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("No suitable non-numeric grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the primary numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there is a grouping field, plot boxplot by group
if 'group_field' in locals():
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the FAIR²-compliant DTI measurement dataset for Autism (derived from ABIDE II) using its Croissant schema and the `mlcroissant` library.
- Explored available distributions and data fields, and extracted key numeric features such as fractional anisotropy (FA), apparent diffusion (AD), etc. All fields and record sets are referenced by their Croissant `@id` as per the schema.
- Performed filtering, normalization, and basic grouping on primary quantitative fields.
- Visualized data distributions and relationships to aid further analysis.

This notebook demonstrates an initial analysis of Croissant-standardized neuroimaging datasets for machine learning applications. For model training or advanced analysis, further domain-specific feature engineering or outlier handling may be required.